# 📘 Домашнє завдання №17. Основи Explainable AI

Виконав: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/DataScience-PBY_HW17

### 📦 Дані
Використати датасет Titanic:
- target: `Survived`
- задача: **бінарна класифікація**

---

## 🔧 Частина 1. Підготовка даних

### 1.1. Препроцесинг
- обробити пропуски  
- закодувати категоріальні змінні  
- обрати релевантні фічі  

---

### 1.2. Feature Engineering (мінімум 2 нові фічі)

Приклади:
- `FamilySize = SibSp + Parch`
- `IsAlone`
- `Title` (з імені)

---

## 🌳 Частина 2. Explainable Boosting Machine (EBM)

### 2.1. Навчання моделі
- використати EBM (InterpretML)

### 2.2. Аналіз глобальних пояснень
- побудувати графіки впливу фіч (shape functions)

### 2.3. Інтерпретація
Для 3–5 фіч:
- як вони впливають на виживання?
- чи є нелінійності?

📌 **Питання:**
- чи є порогові ефекти (наприклад, вік)?
- які фічі найбільш важливі?

---

## 📜 Частина 3. RuleFit

### 3.1. Навчання RuleFit
- підготувати дані (one-hot encoding)
- витягнути правила

### 3.2. Аналіз правил
- вивести топ-10 правил
- пояснити їх

📌 **Питання:**
- які комбінації факторів ведуть до виживання?
- чи є правила, які легко інтерпретувати “людською мовою”?

---

## 📊 Частина 4. SHAP

### 4.1. Модель
- використати будь-яку (наприклад CatBoost / Random Forest)

### 4.2. SHAP аналіз
Побудувати:
- summary plot  
- bar plot  
- dependence plot (мінімум 2 фічі)  
- force або waterfall plot  

### 4.3. Інтерпретація

📌 Відповісти:
- які фічі найважливіші?
- як вони впливають?
- чи є взаємодії?


In [1]:
# Silent installation or update

# Clean cache
!python3 -m pip cache purge -q

# Force updating
package_update = [
    "pip",
    "scikit-learn",
    "pandas",
]

for package_name in package_update:
    !bash -c "python3 -m pip install -U '{package_name}' -q"

# Install missing packages
package_array = [
    "jinja2",
    "ipywidgets",
    "nbformat",
    "kagglehub[pandas-datasets]",
    "numpy",
    "matplotlib",
    "scipy",
    "statsmodels",
    "plotly",
    "joblib",
]

for package_name in package_array:
    !bash -c "python3 -m pip show '{package_name}' > /dev/null 2>&1 || python3 -m pip install -U '{package_name}' -q"


In [2]:
# Synchronization with remote source

import shutil
from pathlib import Path

# Input data
hm_version = 17

# Solution
git_project_url = f"https://github.com/BogdanPinchuk/DataScience-PBY_HW{hm_version}.git"
main_file_name = f"Bohdan_Pinchuk_DS_HW{hm_version}.ipynb"

# upload all files
current_path = !pwd
current_path = current_path[0]
parent_path = !dirname "$current_path"
parent_path = parent_path[0]
temp_path = f"{parent_path}/temp"

# Clone data
!rm -rf "$temp_path"
!git clone "$git_project_url" "$temp_path"

source = Path(temp_path)
destination = Path(current_path)
exclude = {main_file_name, ".git", ".idea"}

for item in source.iterdir():
    if item.name in exclude:
        continue

    target = destination / item.name
    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

# Clean temp folder
!rm -rf "$temp_path"

Cloning into '/Users/bohdanpinchuk/Documents/Data Science/Development/Data_Science/Practical_tasks/Homework_17/temp'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 20 (delta 2), reused 20 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 7.60 KiB | 7.60 MiB/s, done.
Resolving deltas: 100% (2/2), done.


## ✳️ Підготовка датасетів

In [4]:
# Downloading data

import apps.main as mn

# Input data
update_db = False
db_file_name = f"resources/store_hw{hm_version}.db"

# Solution
titanic_dataset = mn.download_and_extract_from_kagglehub("yasserh/titanic-dataset", "Titanic-Dataset.csv",
                                                         db_file_name, update_db)

# Print result
# display(titanic_dataset)


## ✅ Рішення 1.1

- обробити пропуски
- закодувати категоріальні змінні
- обрати релевантні фічі

In [5]:
# Data analysis

import numpy as np
import pandas as pd
import apps.reporter as rpt
from IPython.core.display import Markdown

# Input data
data_set = titanic_dataset
n_columns = data_set.columns.size
n_rows = data_set.index.size

# Solution
columns_df = pd.DataFrame(data_set.columns, columns=["Columns"])
types_df = pd.DataFrame(data_set.dtypes, columns=["Types"])

col_num_type = data_set.select_dtypes(include=np.number).columns
col_cat_type = data_set.select_dtypes(exclude=np.number).columns

empty_val_by_col = data_set.isnull().sum().to_frame(name='count')
is_empty_val = empty_val_by_col.values.sum() > 0

rp = rpt.Reporter()
rp.add_item("Кількість рядків", str(n_rows))
rp.add_item("Кількість об’єктів у датасеті", str(n_rows))
rp.add_item("Кількість стовпців", str(n_columns))
rp.add_item("Кількість числових ознак", str(col_num_type.size))
rp.add_item("Кількість категоріальних ознак", str(col_cat_type.size))
rp.add_item("Пропущені значення", 'є' if is_empty_val > 0 else 'немає')

# Print results
display(Markdown(f"### Аналіз даних"))
rp.print_pd_report("Параметри таблиці до обробки")

if is_empty_val > 0:
    display(empty_val_by_col.style.set_caption("Кількість пропусків"))
display(types_df.style.set_caption("Типи даних"))

display(data_set)

### Аналіз даних

Attribute,Result
Кількість рядків,891
Кількість об’єктів у датасеті,891
Кількість стовпців,12
Кількість числових ознак,7
Кількість категоріальних ознак,5
Пропущені значення,є


,count
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


,Types
PassengerId,int64
Survived,int64
Pclass,int64
Name,str
Sex,str
Age,float64
SibSp,int64
Parch,int64
Ticket,str
Fare,float64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C
